In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency

df = pd.read_csv("cookie_cats.csv")

print("Dataset shape:", df.shape)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'cookie_cats.csv'

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency

df = pd.read_csv("cookie_cats.csv")

print("Dataset shape:", df.shape)
df.head()


FileNotFoundError: [Errno 2] No such file or directory: 'cookie_cats.csv'

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency

df = pd.read_csv("raw cookie_cats.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (90189, 5)


,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


In [4]:
# Retention rates by experiment group

retention_summary = df.groupby("version").agg(
    users=("userid", "count"),
    day1_retention=("retention_1", "mean"),
    day7_retention=("retention_7", "mean"),
    avg_game_rounds=("sum_gamerounds", "mean")
)

retention_summary["day1_retention"] *= 100
retention_summary["day7_retention"] *= 100

retention_summary.round(2)

,users,day1_retention,day7_retention,avg_game_rounds
version,,,,
gate_30,44700,44.82,19.02,52.46
gate_40,45489,44.23,18.20,51.30


In [5]:
gate30_d1 = df[df["version"] == "gate_30"]["retention_1"].mean()
gate40_d1 = df[df["version"] == "gate_40"]["retention_1"].mean()

gate30_d7 = df[df["version"] == "gate_30"]["retention_7"].mean()
gate40_d7 = df[df["version"] == "gate_40"]["retention_7"].mean()

print("Day-1 difference:",
      round((gate40_d1 - gate30_d1) * 100, 2),
      "percentage points")

print("Day-7 difference:",
      round((gate40_d7 - gate30_d7) * 100, 2),
      "percentage points")

Day-1 difference: -0.59 percentage points
Day-7 difference: -0.82 percentage points


In [6]:
# Chi-square test: Day-1 retention

day1_table = pd.crosstab(df["version"], df["retention_1"])

chi2_d1, p_d1, dof_d1, expected_d1 = chi2_contingency(day1_table)

print("Day-1 p-value:", round(p_d1, 4))


# Chi-square test: Day-7 retention

day7_table = pd.crosstab(df["version"], df["retention_7"])

chi2_d7, p_d7, dof_d7, expected_d7 = chi2_contingency(day7_table)

print("Day-7 p-value:", round(p_d7, 4))

Day-1 p-value: 0.0755
Day-7 p-value: 0.0016


In [7]:
# Exclude extreme engagement outlier for engagement analysis

df_clean = df[df["sum_gamerounds"] < 49854].copy()

print("Original rows:", len(df))
print("Clean rows:", len(df_clean))

Original rows: 90189
Clean rows: 90188


In [8]:
correlation = df_clean["sum_gamerounds"].corr(
    df_clean["retention_7"].astype(int)
)

print("Correlation between game rounds and Day-7 retention:",
      round(correlation, 3))

Correlation between game rounds and Day-7 retention: 0.519


In [9]:
df_clean["engagement_quartile"] = pd.qcut(
    df_clean["sum_gamerounds"],
    q=4,
    labels=["Q1 - Low", "Q2", "Q3", "Q4 - High"],
    duplicates="drop"
)

engagement_summary = df_clean.groupby(
    "engagement_quartile",
    observed=True
).agg(
    users=("userid", "count"),
    avg_rounds=("sum_gamerounds", "mean"),
    day7_retention=("retention_7", "mean")
)

engagement_summary["day7_retention"] *= 100

engagement_summary.round(2)

,users,avg_rounds,day7_retention
engagement_quartile,,,
Q1 - Low,24717,2.27,1.27
Q2,20397,10.28,4.13
Q3,22622,30.32,14.01
Q4 - High,22452,163.75,55.47


In [10]:
# Create fields for Tableau

df_clean["retention_1_num"] = df_clean["retention_1"].astype(int)
df_clean["retention_7_num"] = df_clean["retention_7"].astype(int)

df_clean["experiment_group"] = df_clean["version"].map({
    "gate_30": "Gate 30 - Control",
    "gate_40": "Gate 40 - Treatment"
})

# Export cleaned dataset
df_clean.to_csv(
    "cookie_cats_cleaned.csv",
    index=False
)

print("Export complete!")
print(df_clean.shape)

df_clean.head()

Export complete!
(90188, 9)


,userid,version,sum_gamerounds,retention_1,retention_7,engagement_quartile,retention_1_num,retention_7_num,experiment_group
0,116,gate_30,3,False,False,Q1 - Low,0,0,Gate 30 - Control
1,337,gate_30,38,True,False,Q3,1,0,Gate 30 - Control
2,377,gate_40,165,True,False,Q4 - High,1,0,Gate 40 - Treatment
3,483,gate_40,1,False,False,Q1 - Low,0,0,Gate 40 - Treatment
4,488,gate_40,179,True,True,Q4 - High,1,1,Gate 40 - Treatment
